In [3]:
from transformers import pipeline

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
text = "The new data center will create hundreds of jobs but is terrible for climate change."

aspects = ["economy", "environment", "housing", "quality of life", "government", "aesthetics"]

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

def absa_single_step(text, aspect):
    labels = ["positive", "negative", "neutral", "not mentioned"]
    # labels = ["economy", "environment", "housing", "quality of life", "government", "aesthetics"]
    final_labels = []
    final_scores = []

    hypothesis_template = (
        f"The sentiment toward {aspect} is {{}}."
    )

    result = classifier(
        text,
        labels,
        multi_label = True,
        hypothesis_template=hypothesis_template
    )

    return result

    # print(result['labels'])
    # print(result['scores'])

    # for i in range(len(result['labels'])):
    #     if result['scores'][i] > 0.5:
    #         final_labels.append(result['labels'][i])
    #         final_scores.append(result['scores'][i])

    # return final_labels
    # return final_scores

    return {
        "aspect": aspect,
        "label": result["labels"][0],
        "score": result["scores"][0]
    }

for aspect in aspects:
    print(aspect, absa_single_step(text, aspect))


Device set to use mps:0


economy {'sequence': 'The new data center will create hundreds of jobs but is terrible for climate change.', 'labels': ['negative', 'neutral', 'positive', 'not mentioned'], 'scores': [0.8556614518165588, 0.738456130027771, 0.06763278692960739, 0.004673219285905361]}
environment {'sequence': 'The new data center will create hundreds of jobs but is terrible for climate change.', 'labels': ['negative', 'neutral', 'not mentioned', 'positive'], 'scores': [0.9986086487770081, 0.12945576012134552, 0.0015713180182501674, 0.0014738317113369703]}
housing {'sequence': 'The new data center will create hundreds of jobs but is terrible for climate change.', 'labels': ['negative', 'neutral', 'not mentioned', 'positive'], 'scores': [0.2484268844127655, 0.07711925357580185, 0.033049147576093674, 0.007014215923845768]}
quality of life {'sequence': 'The new data center will create hundreds of jobs but is terrible for climate change.', 'labels': ['negative', 'neutral', 'not mentioned', 'positive'], 'score

In [3]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

text = "The pasta was delicious but the service was too slow."
candidate_labels = [
    "food quality", "food taste", "service speed", "staff friendliness"
]
# To get sentiment, you can include it in the label:
# candidate_labels = ["food was good", "food was bad", "service was fast", "service was slow"]

# Use multi_label=True if multiple aspects are expected
result = classifier(text, candidate_labels, multi_label=True)
print(result)

Device set to use mps:0


{'sequence': 'The pasta was delicious but the service was too slow.', 'labels': ['food taste', 'food quality', 'service speed', 'staff friendliness'], 'scores': [0.9755818247795105, 0.9364194869995117, 0.5148895978927612, 0.004734459333121777]}


In [33]:
text = "The food was delicious, but the service was incredibly slow."
aspects = ["food", "service", "ambiance"]

# 1. Detect which aspects are mentioned
aspect_results = classifier(
    text,
    aspects,
    multi_label=True
)
print(aspect_results)
final_labels = []
final_scores = []

for label in aspect_results['labels']:
    element = aspect_results['labels'].index(label)
    if aspect_results['scores'][element] > 0.5:
        final_labels.append(label)
        final_scores.append(aspect_results['scores'][element])

print(f"Detected Aspects: {final_labels}")

# 2. Analyze sentiment for a specific aspect
# Use a specific hypothesis template for better accuracy
sentiment_labels = ["positive", "negative", "neutral"]

for theme in final_labels:
    sentiment_results = classifier(
        text, 
        sentiment_labels, 
        hypothesis_template=f"The sentiment towards {theme} is {{}}."
    )

    print(f"{theme} sentiment: {sentiment_results['labels'][0]} ({sentiment_results['scores'][0]})")

{'sequence': 'The food was delicious, but the service was incredibly slow.', 'labels': ['food', 'service', 'ambiance'], 'scores': [0.9949820637702942, 0.6946498155593872, 0.00882975198328495]}
Detected Aspects: ['food', 'service']
food sentiment: positive (0.9337558150291443)
service sentiment: negative (0.8964154124259949)


In [38]:
text = "The food was delicious, but the service was incredibly slow."
aspects = ["food", "service", "ambiance"]

# Analyze sentiment for a specific aspect
# Use a specific hypothesis template for better accuracy
sentiment_labels = ["positive", "negative", "neutral", "theme not present"]

for theme in aspects:
    sentiment_results = classifier(
        text, 
        sentiment_labels, 
        hypothesis_template=f"The sentiment towards {theme} is {{}}."
    )

    print(f"{theme} sentiment: {sentiment_results['labels'][0]} ({sentiment_results['scores'][0]})")

food sentiment: positive (0.9225658178329468)
service sentiment: negative (0.642129123210907)
ambiance sentiment: neutral (0.48503631353378296)


In [ ]:
text = "The new data center will create hundreds of well-paying jobs but is terrible for native flora and fauna."
themes = ["infrastructure and utilities", "housing prices", "economy and jobs", "quality of life (noise, light pollution)", "environment and climate change", "visual impact of datacenters", "government (policies, laws)"]

# 1. Detect which themes are mentioned
aspect_results = classifier(
    text,
    themes,
    multi_label=True
)
print(aspect_results)
final_labels = []
final_scores = []

for label in aspect_results['labels']:
    element = aspect_results['labels'].index(label)
    if aspect_results['scores'][element] > 0.5:
        final_labels.append(label)
        final_scores.append(aspect_results['scores'][element])

print(f"Detected Aspects: {final_labels}")

# 2. Analyze sentiment for a specific aspect
# Use a specific hypothesis template for better accuracy
sentiment_labels = ["positive", "negative", "neutral"]

for theme in final_labels:
    sentiment_results = classifier(
        text, 
        sentiment_labels, 
        hypothesis_template=f"The sentiment towards {theme} is {{}}."
    )

    print(f"{theme} sentiment: {sentiment_results['labels'][0]} ({sentiment_results['scores'][0]})")

{'sequence': 'The new data center will create hundreds of well-paying jobs but is terrible for native flora and fauna.', 'labels': ['visual impact of datacenters', 'environment and climate change', 'economy and jobs', 'quality of life (noise, light pollution)', 'infrastructure and utilities', 'government (policies, laws)', 'housing prices'], 'scores': [0.9059009552001953, 0.8824754357337952, 0.522346556186676, 0.03415374830365181, 0.0005095463711768389, 0.00021433406800497323, 5.481329935719259e-05]}
Detected Aspects: ['visual impact of datacenters', 'environment and climate change', 'economy and jobs']
visual impact of datacenters sentiment: negative (0.9409764409065247)
environment and climate change sentiment: negative (0.8126057386398315)
economy and jobs sentiment: neutral (0.573358952999115)
